Extract RGB Data

In [1]:
%pip install pandas

  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB 435.7 kB/s eta 0:00:23
    --------------------------------------- 0.1/9.7 MB 901.1 kB/s eta 0:00:11
    --------------------------------------- 0.2/9.7 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.3/9.7 MB 1.2 MB/s eta 0:00:08
   - -------------------------------------- 0.4/9.7 MB 1.6 MB/s eta 0:00:06
   -- ------------------------------------- 0.7/9.7 MB 2.2 MB/s eta 0:00:05
   --- ------------------------------------ 0.9/9.7 MB 2.5 MB/s eta 0:00:04
   ---- ----------------------------------- 1.1/9.7 MB 2.8 MB/s eta 0:00:04
   ----- ---------------------------------- 1.4/9.7 MB 3.1 MB/s eta 0:00:03
   ------ --------------------------------- 1.5/9.7 MB 3.2 MB/s eta 0:00:03
   ------ --------------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import json

In [3]:
train_folder = Path("../data/raw/train") #setting the path to the training data folder

In [4]:
annotation_path = Path("../data/raw/train/_annotations.coco.json")

with open(annotation_path, "r") as file:
    coco_data = json.load(file)

defect_types = {}
# Create a lookup from category ID to category name
category_names = {}

for category in coco_data["categories"]:
    category_names[category["id"]] = category["name"]
    
for annotation in coco_data["annotations"]:
    image_id = annotation["image_id"]
    category_id = annotation["category_id"]
    defect_name = category_names[category_id]

    if image_id not in defect_types:
        defect_types[image_id] = []

    defect_types[image_id].append(defect_name)

image_labels = []

for image_record in coco_data["images"]:

    image_id = image_record["id"]
    filename = image_record["file_name"]

    defects = defect_types.get(image_id, [])
    number_of_defects = len(defects)

    if number_of_defects > 0:
        label = "FAIL"
    else:
        label = "PASS"

    image_labels.append({
        "image_id": image_id,
        "filename": filename,
        "number_of_defects": number_of_defects,
        "defect_types": defects,
        "label": label
    })

labels_df = pd.DataFrame(image_labels)

In [5]:
image_features = []

for _, row in labels_df.iterrows():

    image_path = train_folder / row["filename"]

    image = Image.open(image_path).convert("RGB")

    image_array = np.array(image)

    mean_rgb = image_array.mean(axis=(0, 1))
    std_rgb = image_array.std(axis=(0, 1))

    image_features.append({
        "filename": row["filename"],
        "mean_red": mean_rgb[0],
        "mean_green": mean_rgb[1],
        "mean_blue": mean_rgb[2],
        "std_red": std_rgb[0],
        "std_green": std_rgb[1],
        "std_blue": std_rgb[2],
        "label": row["label"]
    })

    features_df = pd.DataFrame(image_features)

features_df.head()

,filename,mean_red,mean_green,mean_blue,std_red,std_green,std_blue,label
0,frame_with_red_8214_jpg.rf.5acda82172bcf022bd7...,30.301752,28.310728,28.183426,37.372829,28.631737,29.002394,FAIL
1,frame_with_red_1237_jpg.rf.7c920cb9b9d7c993551...,29.648264,30.121911,28.617816,40.769531,29.050063,28.730397,FAIL
2,frame_with_red_6884_jpg.rf.73d4cccf385f78c6473...,29.461821,27.114720,27.660437,40.344334,29.204149,29.422255,FAIL
3,frame_with_red_2737_jpg.rf.8af748cd8904c67c7a0...,49.706422,45.106337,46.502819,41.100140,27.931970,28.625920,PASS
4,frame_with_red_10465_jpg.rf.800ed5e3f0775757ce...,56.781209,52.052703,50.397927,52.871247,40.003867,39.245673,FAIL


In [6]:
output_path = Path("../data/processed/train_features.csv") #output path for the features CSV file

features_df.to_csv(output_path, index=False)
print("Saved to:", output_path)
print("File exists:", output_path.exists())

Saved to: ..\data\processed\train_features.csv
File exists: True
